In [406]:
import numpy as np
from tqdm import tqdm

In [407]:
T = int(1e4)
N = 1000
r = 5
p = 200
q = 200
penalty = 1e0/np.sqrt(N)
tol = 1e-4

# generate features
Z = np.random.normal(loc=0, scale=1, size=(N, p))
X = np.random.normal(loc=0, scale=1, size=(N, q))
row_norms_Z = np.linalg.norm(Z, axis=1, keepdims=True)
row_norms_X = np.linalg.norm(X, axis=1, keepdims=True)
Z = Z / row_norms_Z
X = X / row_norms_X
KZ = np.dot(Z, Z.T)
KX = np.dot(X, X.T)

# generate random projection
P = np.random.normal(loc=0, scale=1, size=(p, q))
L, S, Rt = np.linalg.svd(P, full_matrices=False)
Lr = L[:, :r]
Rr = Rt[:r, :].T

# generate the outcome
Zr = np.dot(Z, Lr) # n*r
Xr = np.dot(X, Rr) # n*r
y = np.diag(np.dot(Zr, Xr.T))

In [408]:
def DKRL_primal(Z, X, y, r, penalty, tol, T):
    N = Z.shape[0]

    Ut = np.random.normal(loc=0, scale=1, size=(N, r))
    Vt = np.random.normal(loc=0, scale=1, size=(N, r))
    Utt = Ut.copy()
    Vtt = Vt.copy()

    for iter in tqdm(range(T)):
        # update U matrix
        VKX = np.repeat(np.dot(Vt.T, KX), repeats=N, axis=0)
        KZZ = np.tile(KZ, (1, r)) # repeat KZ r times, stack as column
        DesignVt = VKX.T * KZZ  # n * (nr)

        hhat = np.dot(KX, Vtt)
        for i in range(r):
            # Compute residuals first
            ghat = np.dot(KZ, Utt)
            DKZ = DesignVt[:, i * N:(i + 1) * N]
            yhat = (ghat * hhat).sum(axis=1)
            y_residual = y - yhat

            # Update Utt[:, i]
            dUtti = np.linalg.solve(np.dot(DKZ.T, DKZ) + penalty * KZ + 1e-2*np.eye(N), np.dot(DKZ.T, y_residual))
            Utt[:, i] = Utt[:, i] + dUtti
        
        # update V matrix
        UKZ = np.repeat(np.dot(Utt.T, KZ), repeats=N, axis=0)
        KXX = np.tile(KX, (1, r)) 
        DesignUt = UKZ.T * KXX  # n * (nr)
        
        ghat = np.dot(KZ, Utt)
        for i in range(r):
            # Compute residuals first
            hhat = np.dot(KX, Vtt)
            yhat = (ghat * hhat).sum(axis=1)
            y_residual = y - yhat
            
            # Update Vtt[:, i]
            DKX = DesignUt[:, i * N:(i + 1) * N]
            dVtti = np.linalg.solve(np.dot(DKX.T, DKX) + penalty * KX + 1e-2*np.eye(N), np.dot(DKX.T, y_residual))
            Vtt[:, i] = Vtt[:, i] + dVtti

        # check stopping criterion
        # print(iter)
        # print(np.linalg.norm(Utt - Ut)/(np.linalg.norm(Ut) + 1e-4))
        if (np.linalg.norm(Utt - Ut)/(np.linalg.norm(Ut) + 1e-4) < tol and np.linalg.norm(Vtt - Vt)/(np.linalg.norm(Vt) + 1e-3) < tol):
            break

        # if not stop, update Ut and Vt
        Ut = Utt.copy()
        Vt = Vtt.copy()
    
    y_pred = (np.dot(KZ, Utt) * np.dot(KX, Vtt)).sum(axis=1)
    return Utt, Vtt, y_pred
    

In [409]:
def DKRL_dual(Z, X, y, r, penalty, tol, T):
    p = Z.shape[1]
    q = X.shape[1]
    N = Z.shape[0]
    
    Ut = np.random.normal(loc=0, scale=1, size=(p, r))
    Vt = np.random.normal(loc=0, scale=1, size=(q, r))
    Utt = Ut.copy()
    Vtt = Vt.copy()

    for iter in tqdm(range(T)):
        # update U matrix    
        hhat = np.dot(X, Vtt)
        for i in range(r):
            # Compute residuals first
            ghat = np.dot(Z, Utt)
            y_pred = (ghat * hhat).sum(axis=1)
            y_residual = y - y_pred
            DZ = Z * hhat[:, i][:, np.newaxis]

            # Update Utt[:, i]
            dUtti = np.linalg.solve(np.dot(DZ.T, DZ) + penalty * np.eye(p), np.dot(DZ.T, y_residual))
            Utt[:, i] = Utt[:, i] + dUtti
        
        # update V matrix
        ghat = np.dot(Z, Utt)
        for i in range(r):
            # Compute residuals first
            hhat = np.dot(X, Vtt)
            y_pred = (ghat * hhat).sum(axis=1)
            y_residual = y - y_pred
            DX = X * ghat[:, i][:, np.newaxis]
            
            # Update Vtt[:, i]
            dVtti = np.linalg.solve(np.dot(DX.T, DX) + penalty * np.eye(q), np.dot(DX.T, y_residual))
            Vtt[:, i] = Vtt[:, i] + dVtti

        # check stopping criterion
        # print(iter)
        # print(np.linalg.norm(Utt - Ut)/(np.linalg.norm(Ut) + 1e-4))
        if (np.linalg.norm(Utt - Ut)/(np.linalg.norm(Ut) + 1e-4) < tol and np.linalg.norm(Vtt - Vt)/(np.linalg.norm(Vt) + 1e-3) < tol):
            break

        # if not stop, update Ut and Vt
        Ut = Utt.copy()
        Vt = Vtt.copy()

    y_pred = (np.dot(Z, Utt) * np.dot(X, Vtt)).sum(axis=1)
    return Utt, Vtt, y_pred

In [396]:
def DKRL_pred_primal(U, V, KZ_pred, KX_pred):
    y_pred = np.dot(KZ_pred.T, U) * np.dot(KX_pred.T, V)
    y_pred = y_pred.sum(axis=1)
    print(y_pred.shape)
    return y_pred

In [363]:
def DKRL_pred_dual(U, V, Z, X):
    y_pred = np.dot(Z, U) * np.dot(X, V)
    y_pred = y_pred.sum(axis=1)
    return y_pred

In [410]:
# test primal version
U, V, y_pred = DKRL_primal(KZ, KX, y, r = 15, penalty = 1/np.sqrt(N), tol = 1e-4, T = 1000)

y_pred_primal = DKRL_pred_primal(U, V, KZ, KX)

np.linalg.norm(y - y_pred_primal)/np.linalg.norm(y)

  0%|          | 3/1000 [00:01<10:37,  1.56it/s]

(1000,)


np.float64(0.004813313796358122)

In [412]:
# test dual version
U, V, y_pred = DKRL_dual(Z, X, y, r = 10, penalty = 1/np.sqrt(N), tol = 1e-4, T = 10000)

y_pred_dual = DKRL_pred_dual(U, V, Z, X)

np.linalg.norm(y - y_pred)/np.linalg.norm(y)

  0%|          | 5/10000 [00:00<02:37, 63.65it/s]


np.float64(0.0037394704222066416)

In [ ]:
# Below are testing codes

In [371]:
# A dual version of DKRL

Ut = np.random.normal(loc=0, scale=1, size=(p, r))
Vt = np.random.normal(loc=0, scale=1, size=(q, r))
Utt = Ut.copy()
Vtt = Vt.copy()

for iter in tqdm(range(T)):
    # update U matrix    
    hhat = np.dot(X, Vtt)
    for i in range(r):
        # Compute residuals first
        ghat = np.dot(Z, Utt)
        y_pred = (ghat * hhat).sum(axis=1)
        y_residual = y - y_pred
        DZ = Z * hhat[:, i][:, np.newaxis]

        # Update Utt[:, i]
        dUtti = np.linalg.solve(np.dot(DZ.T, DZ) + penalty * np.eye(p), np.dot(DZ.T, y_residual))
        Utt[:, i] = Utt[:, i] + dUtti
    
    # update V matrix
    ghat = np.dot(Z, Utt)
    for i in range(r):
        # Compute residuals first
        hhat = np.dot(X, Vtt)
        y_pred = (ghat * hhat).sum(axis=1)
        y_residual = y - y_pred
        DX = X * ghat[:, i][:, np.newaxis]
        
        # Update Vtt[:, i]
        dVtti = np.linalg.solve(np.dot(DX.T, DX) + penalty * np.eye(q), np.dot(DX.T, y_residual))
        Vtt[:, i] = Vtt[:, i] + dVtti

    # check stopping criterion
    # print(iter)
    # print(np.linalg.norm(Utt - Ut)/(np.linalg.norm(Ut) + 1e-4))
    if (np.linalg.norm(Utt - Ut)/(np.linalg.norm(Ut) + 1e-4) < tol and np.linalg.norm(Vtt - Vt)/(np.linalg.norm(Vt) + 1e-3) < tol):
        break

    # if not stop, update Ut and Vt
    Ut = Utt.copy()
    Vt = Vtt.copy()

 23%|██▎       | 2293/10000 [00:05<00:18, 411.27it/s]


In [394]:
np.linalg.norm(y_residual)/np.linalg.norm(y)

np.float64(0.3364813130127791)

In [393]:
r = 10
penalty = 1e-1
tol = 1e-4
Ut = np.random.normal(loc=0, scale=1, size=(N, r))
Vt = np.random.normal(loc=0, scale=1, size=(N, r))
Utt = Ut.copy()
Vtt = Vt.copy()

for iter in tqdm(range(T)):
    # update U matrix
    VKX = np.repeat(np.dot(Vt.T, KX), repeats=N, axis=0)
    KZZ = np.tile(KZ, (1, r)) # repeat KZ r times, stack as column
    DesignVt = VKX.T * KZZ  # n * (nr)

    hhat = np.dot(KX, Vtt)
    for i in range(r):
        # Compute residuals first
        ghat = np.dot(KZ, Utt)
        DKZ = DesignVt[:, i * N:(i + 1) * N]
        yhat = (ghat * hhat).sum(axis=1)
        y_residual = y - yhat

        # Update Utt[:, i]
        dUtti = np.linalg.solve(np.dot(DKZ.T, DKZ) + penalty * KZ + 1e-2*np.eye(N), np.dot(DKZ.T, y_residual))
        Utt[:, i] = Utt[:, i] + dUtti
    
    # update V matrix
    UKZ = np.repeat(np.dot(Utt.T, KZ), repeats=N, axis=0)
    KXX = np.tile(KX, (1, r)) 
    DesignUt = UKZ.T * KXX  # n * (nr)
    
    ghat = np.dot(KZ, Utt)
    for i in range(r):
        # Compute residuals first
        hhat = np.dot(KX, Vtt)
        yhat = (ghat * hhat).sum(axis=1)
        y_residual = y - yhat
        
        # Update Vtt[:, i]
        DKX = DesignUt[:, i * N:(i + 1) * N]
        dVtti = np.linalg.solve(np.dot(DKX.T, DKX) + penalty * KX + 1e-2*np.eye(N), np.dot(DKX.T, y_residual))
        Vtt[:, i] = Vtt[:, i] + dVtti

    # check stopping criterion
    # print(iter)
    # print(np.linalg.norm(Utt - Ut)/(np.linalg.norm(Ut) + 1e-4))
    if (np.linalg.norm(Utt - Ut)/(np.linalg.norm(Ut) + 1e-4) < tol and np.linalg.norm(Vtt - Vt)/(np.linalg.norm(Vt) + 1e-3) < tol):
        break

    # if not stop, update Ut and Vt
    Ut = Utt.copy()
    Vt = Vtt.copy()

  0%|          | 18/10000 [00:06<57:33,  2.89it/s]


In [389]:
np.linalg.norm(y_residual)/np.linalg.norm(y)

np.float64(0.3463790972790391)

# test codes:

In [257]:
# compute the kernel matrices of Z and X
# KZ = 
# KX = 
# y = 
r = 5
T = int(1e3)
penalty = 1e-2
tol = 1e-3
Ut = np.random.normal(loc=0, scale=1, size=(N, r))
Vt = np.random.normal(loc=0, scale=1, size=(N, r))

for iter in tqdm(range(T)):
    # update U matrix
    VKX = np.repeat(np.dot(Vt.T, KX), repeats=N, axis=0)
    KZZ = np.tile(KZ, (1, r)) # repeat KZ r times, stack as column
    DesignVt = VKX.T * KZZ  # n * (nr)
    Utt = np.linalg.solve(np.dot(DesignVt.T, DesignVt) + penalty * np.kron(np.eye(r, dtype=int), KZ), np.dot(DesignVt.T, y)) 
    # Utt = np.linalg.solve(np.dot(DesignVt.T, DesignVt) + penalty * np.eye(N*r), np.dot(DesignVt.T, y)) 
    Utt = Utt.reshape((N, r), order="F")

    # update V matrix
    UKZ = np.repeat(np.dot(Utt.T, KZ), repeats=N, axis=0)
    KXX = np.tile(KX, (1, r)) 
    DesignUt = UKZ.T * KXX  # n * (nr)
    Vtt = np.linalg.solve(np.dot(DesignUt.T, DesignUt) + penalty * np.kron(np.eye(r, dtype=int), KX), np.dot(DesignUt.T, y))
    # Vtt = np.linalg.solve(np.dot(DesignUt.T, DesignUt) + penalty * np.eye(N*r), np.dot(DesignUt.T, y))
    Vtt = Vtt.reshape((N, r), order="F")

    # check stopping criterion
    # print(iter)
    # print(np.linalg.norm(Utt - Ut)/(np.linalg.norm(Ut) + 1e-4))
    if (np.linalg.norm(Utt - Ut)/(np.linalg.norm(Ut) + 1e-4) < tol and np.linalg.norm(Vtt - Vt)/(np.linalg.norm(Vt) + 1e-3) < tol):
        break

    # if not stop, update Ut and Vt
    Ut = Utt
    Vt = Vtt

 79%|███████▉  | 793/1000 [14:59<03:54,  1.13s/it]  


KeyboardInterrupt: 

In [258]:
np.linalg.norm(y_residual)/np.linalg.norm(y)

np.float64(0.7451040825557838)